In [ ]:
!pip install groq chromadb sentence-transformers pandas -q

import pandas as pd
import sqlite3
from groq import Groq
import chromadb
import os

GROQ_API_KEY = "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"
client =Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.1-8b-instant"
print("Groq client configured.")
print(f"Model:{MODEL}")
print("Status: Ready to generate AI responses")

responsible_system_prompt="""
You are a helpful data analysis assistant.
You only answer questions based on the data provided to you.
If you are not sure about something, say: I do not have enough information to answer that accurately.
Never make up statistics or facts that are not in the data you receive.
"""
test_response=client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role":"system","content":responsible_system_prompt},
        {"role":"user","content":"What is the population of Mars?"}
    ]
)

Groq client configured.
Model:llama-3.1-8b-instant
Status: Ready to generate AI responses


In [ ]:
student_df= pd.read_csv("student_performance.csv")

print("===Student performance dataset===")
print(f"Shape:{student_df.shape}")
print(student_df.head())

===Student performance dataset===
Shape:(30, 11)
   student_id          name  age  gender branch  attendance_pct  \
0           1  Aarav Sharma   20    Male    CSE              85   
1           2   Priya Patel   21  Female    ECE              92   
2           3   Rohit Kumar   20    Male   MECH              67   
3           4    Sneha Iyer   22  Female    CSE              95   
4           5  Vikram Singh   21    Male  CIVIL              72   

   assignment_score  midterm_score  final_score  gpa passed  
0                78             72           76  7.6    Yes  
1                88             85           89  8.9    Yes  
2                55             60           58  5.8    Yes  
3                92             90           94  9.4    Yes  
4                62             65           63  6.3    Yes  


In [ ]:
notes_df=pd.read_csv("college_notes.csv")

print("===College notes dataset===")
print(f"Shape:{notes_df.shape}")
print(notes_df.head())

===College notes dataset===
Shape:(15, 4)
  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N005  Data Engineering      Big Data and PySpark   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Big Data refers to extremely large datasets th...  


In [ ]:
print("Data quality Report:")
print("Missing values per column")
print(student_df.isna().sum())
print()

print("Data types:")
print(student_df.dtypes)
print()


Data quality Report:
Missing values per column
student_id          0
name                0
age                 0
gender              0
branch              0
attendance_pct      0
assignment_score    0
midterm_score       0
final_score         0
gpa                 0
passed              0
dtype: int64

Data types:
student_id            int64
name                 object
age                   int64
gender               object
branch               object
attendance_pct        int64
assignment_score      int64
midterm_score         int64
final_score           int64
gpa                 float64
passed               object
dtype: object



In [ ]:
conn=sqlite3.connect("memory")
student_df.to_sql('students', conn, if_exists='replace', index=False)
query1 = """SELECT branch, COUNT(*) AS total_students, ROUND(AVG(gpa), 2) AS avg_gpa, ROUND(AVG(attendance_pct), 2) AS avg_attendance FROM students GROUP BY branch ORDER BY avg_gpa DESC"""
branch_analysis=pd.read_sql(query1,conn)
print("=== Branch-wise Performance Analysis ===")
print(branch_analysis)

=== Branch-wise Performance Analysis ===
  branch  total_students  avg_gpa  avg_attendance
0     IT               5     8.64           89.40
1    CSE              10     7.42           80.00
2   MECH               5     7.22           79.40
3  CIVIL               4     6.75           75.00
4    ECE               6     6.38           69.17


In [ ]:
query2="""SELECT name,branch,gpa,attendance_pct,passed
FROM students
ORDER BY gpa DESC
LIMIT 5
"""

top_students=pd.read_sql(query2,conn)
print("=== Top 5 Students ===")
print(top_students)


=== Top 5 Students ===
               name branch  gpa  attendance_pct passed
0    Meera Krishnan     IT  9.5              96    Yes
1        Sneha Iyer    CSE  9.4              95    Yes
2  Lakshmi Chandran    CSE  9.2              94    Yes
3      Swathi Menon     IT  9.1              93    Yes
4       Priya Patel    ECE  8.9              92    Yes


In [ ]:
query3 = """
SELECT
    passed,
    COUNT(*) AS student_count,
    ROUND(AVG(gpa), 2) AS avg_gpa
FROM students
GROUP BY passed
"""
pass_fail = pd.read_sql(query3,conn)
print("=== Pass/Fail Statistics ===")
print(pass_fail.to_string(index=False))
total = len(student_df)
passed = len(student_df[student_df['passed'] == 'Yes'])
pass_rate = round((passed / total) * 100, 1)
print(f"\nOverall Pass Rate: {pass_rate}%({passed}/{total} students)")

=== Pass/Fail Statistics ===
passed  student_count  avg_gpa
    No              3     4.47
   Yes             27     7.61

Overall Pass Rate: 90.0%(27/30 students)


In [ ]:
branch_summary = ""
for _,row in branch_analysis.iterrows():
   branch_summary += f"-{row['branch']}:{row['total_students']} students,Avg GPA{row['avg_gpa']},Avg Attendance{row['avg_attendance']}%\n"
top_summary = ""
for _, row in top_students.iterrows():
  top_summary   += f"-{row['name']}({row['branch']}):GPA{row['gpa']}\n"
data_summary = f"""
 STUDENT PERFORMANCE DATA SUMMARY
 Total Students:{total}
 Overall Pass Rate: {pass_rate}%
 Average GPA across all students: {round(student_df['gpa'].mean(),2)}

 Performance by Branch:
 {branch_summary}

 Top 5 Students by GPA:
 {top_summary}
 """
print("Data Summary prepared ===")
print(data_summary)

Data Summary prepared ===

 STUDENT PERFORMANCE DATA SUMMARY
 Total Students:30
 Overall Pass Rate: 90.0%
 Average GPA across all students: 7.29

 Performance by Branch:
 -IT:5 students,Avg GPA8.64,Avg Attendance89.4%
-CSE:10 students,Avg GPA7.42,Avg Attendance80.0%
-MECH:5 students,Avg GPA7.22,Avg Attendance79.4%
-CIVIL:4 students,Avg GPA6.75,Avg Attendance75.0%
-ECE:6 students,Avg GPA6.38,Avg Attendance69.17%


 Top 5 Students by GPA:
 -Meera Krishnan(IT):GPA9.5
-Sneha Iyer(CSE):GPA9.4
-Lakshmi Chandran(CSE):GPA9.2
-Swathi Menon(IT):GPA9.1
-Priya Patel(ECE):GPA8.9

 
